# User-Defined Functions

Functions turn repeated logic into named, reusable operations. A good function has a clear purpose, a descriptive name, documented parameters, and a single responsibility.

## Learning goals

- Define functions with positional, keyword, and default arguments.
- Write useful docstrings and type hints.
- Explain local and global scope.
- Implement the sigmoid function.
- Implement Newton's method for the square root.


## 1. Anatomy of a function

A function definition contains the `def` keyword, a name, a parameter list, a colon, an indented body, an optional docstring, and an optional `return` statement. The name should be lowercase with underscores and should usually begin with a verb.


In [ ]:
def compute_rectangle_area(height: float, width: float) -> float:
    """Return the area of a rectangle.

    Args:
        height: The height of the rectangle.
        width: The width of the rectangle.

    Returns:
        The area, computed as height multiplied by width.
    """
    return height * width


print(compute_rectangle_area(5, 6))
print(compute_rectangle_area(height=5, width=6))
print(compute_rectangle_area(width=6, height=5))

## 2. Default and keyword arguments

Default values make parameters optional. Keyword arguments make calls self-documenting and allow arguments to be supplied out of order. Avoid mutable defaults such as `items=[]`; use `None` and create a new object inside the function when needed.


In [ ]:
def compute_rectangle_area_with_defaults(
    height: float = 0.0,
    width: float = 0.0,
) -> float:
    """Return the area of a rectangle, using zero for omitted dimensions."""
    return height * width


print(compute_rectangle_area_with_defaults())
print(compute_rectangle_area_with_defaults(5, 6))
print(compute_rectangle_area_with_defaults(width=6))

## 3. The sigmoid function

The logistic sigmoid is a smooth, increasing function that maps any real number into $(0, 1)$:

$$
\sigma(x) = \frac{1}{1 + e^{-x}}.
$$

It is used as an activation function and as the inverse of the log-odds. Unlike the original teaching example, the function below takes `x` as a parameter instead of reading a global variable. Explicit parameters make dependencies visible and the function testable.


In [ ]:
import math


def compute_sigmoid(x: float) -> float:
    """Return the logistic sigmoid of x."""
    return 1.0 / (1.0 + math.exp(-x))


for value in (-2.0, 0.0, 2.0):
    print(f"sigmoid({value:+.1f}) = {compute_sigmoid(value):.6f}")

print("sigmoid(0) =", compute_sigmoid(0.0))

The sigmoid is stable for moderate inputs. For very large negative values, `math.exp(-x)` can overflow; the numerical-stability notebook shows a stable evaluation strategy.


## 4. Scope

Python resolves names using the local, enclosing, global, and built-in scopes, often summarized as LEGB. Assignment inside a function creates a local name unless the name is declared `global` or `nonlocal`.

A function can read a global variable. A local assignment with the same name shadows the global one and does not modify it.


In [ ]:
global_multiplier = 4


def scale(value: float) -> float:
    """Multiply value by the module-level multiplier."""
    return value * global_multiplier


def scale_with_local_shadow(value: float) -> float:
    """Use a local multiplier without changing the global one."""
    global_multiplier = 3
    return value * global_multiplier


print(scale(2.0))
print(scale_with_local_shadow(2.0))
print(global_multiplier)

A name that is assigned inside a function is local for the entire function body. Reading it before the assignment raises `UnboundLocalError`.


In [ ]:
def broken_scope() -> float:
    result = value * 2  # noqa: F823
    value = 1  # noqa: F841
    return result


try:
    broken_scope()
except UnboundLocalError as error:
    print(type(error).__name__, error)

## 5. Newton's method for the square root

Newton's method solves $f(x) = 0$ by repeatedly following the tangent line:

$$
x_{n+1} = x_n - \frac{f(x_n)}{f'(x_n)}.
$$

For $f(x) = x^2 - N$, the update becomes

$$
x_{n+1} = \frac{1}{2}\left(x_n + \frac{N}{x_n}\right).
$$

The method converges quadratically near the root. A tolerance stops the loop when the estimate stops changing meaningfully; a maximum iteration count prevents infinite loops.


In [ ]:
def newton_sqrt(
    number: float,
    tolerance: float = 1e-12,
    max_iterations: int = 100,
) -> float:
    """Return the square root of number using Newton's method.

    Args:
        number: A non-negative number.
        tolerance: Stop when the change between estimates is below this value.
        max_iterations: Maximum number of updates.

    Raises:
        ValueError: If number is negative.
    """
    if number < 0:
        raise ValueError("number must be non-negative")
    if number == 0:
        return 0.0

    estimate = number / 2.0
    for _ in range(max_iterations):
        next_estimate = 0.5 * (estimate + number / estimate)
        if abs(next_estimate - estimate) < tolerance:
            return next_estimate
        estimate = next_estimate

    return estimate


for value in (2.0, 4.0, 10.0):
    approximation = newton_sqrt(value)
    print(f"sqrt({value}) ~= {approximation:.12f}")
    print("absolute error:", abs(approximation - math.sqrt(value)))

## Practice

1. Write `compute_circle_area(radius)` with a docstring and a type hint.
2. Write `celsius_to_fahrenheit(celsius)` and its inverse, then verify that they round-trip.
3. Add a `max_iterations` guard to a loop that sums a series.
4. Explain why `scale_with_local_shadow` does not change `global_multiplier`.
5. Modify `newton_sqrt` to return the number of iterations as well as the estimate.
